# NBA Player Value Analyzer
## Notebook 04: Developing a Salary-Adjusted Player Value Model

### Research Question
Which NBA players provide the greatest on-court statistical
production relative to their 2024–25 salaries?

### Objectives
1. Standardize player statistics to account for playing time.
2. Create a weighted production score.
3. Model expected production using player salaries.
4. Calculate salary-adjusted player value scores.
5. Rank players by their relative value.
6. Evaluate the model and identify limitations.

### Methodology
The analysis focuses on players who accumulated at least
approximately 1,000 regular-season minutes.

The production model incorporates scoring, playmaking,
rebounding, defensive box-score statistics, and efficiency.

Linear regression on log salary is used to estimate
expected production. A player's value score is the
difference between actual and expected production.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

file_path = (
    project_dir /
    "data/processed/nba_player_values_base.csv"
)

df = pd.read_csv(file_path)

print("Total players:", len(df))
print("Players with salaries:",
      df["salary_2024_25"].notna().sum())

display(df.head())

Total players: 569
Players with salaries: 483


,player_id,player_name,team,age,games_played,minutes_per_game,points_per_game,rebounds_per_game,assists_per_game,steals_per_game,...,FGA,FG3M,FG3A,FTM,FTA,TEAM_COUNT,TS_PCT,salary_2024_25,salary_teams,total_minutes_est
0,1630639,A.J. Lawson,TOR,24.0,26,18.7,9.1,3.3,1.2,0.5,...,7.3,1.3,3.9,1.7,2.4,1,0.541981,100000.0,TOR,486.2
1,1631260,AJ Green,MIL,25.0,73,22.7,7.4,2.4,1.5,0.5,...,5.8,2.1,5.0,0.3,0.4,1,0.620584,2120693.0,MIL,1657.1
2,1642358,AJ Johnson,WAS,20.0,29,22.0,7.6,2.0,2.6,0.4,...,7.3,0.8,3.1,1.1,1.3,2,0.479763,2795294.0,WAS,638.0
3,203932,Aaron Gordon,DEN,29.0,51,28.4,14.7,4.8,3.2,0.5,...,9.7,1.5,3.4,2.8,3.5,1,0.649576,22841455.0,DEN,1448.4
4,1628988,Aaron Holiday,HOU,28.0,62,12.8,5.5,1.3,1.3,0.3,...,4.3,1.2,2.9,0.5,0.7,1,0.594322,4668000.0,HOU,793.6


In [3]:
# Approximate season minutes
df["total_minutes_est"] = (
    df["games_played"] *
    df["minutes_per_game"]
)

# Select eligible players
players = df[
    (df["total_minutes_est"] >= 1000) &
    (df["salary_2024_25"].notna()) &
    (df["salary_2024_25"] > 0)
].copy()

# Convert salary to millions
players["salary_millions"] = (
    players["salary_2024_25"] / 1_000_000
)

model_columns = [
    "points_per_game",
    "rebounds_per_game",
    "assists_per_game",
    "steals_per_game",
    "blocks_per_game",
    "minutes_per_game",
    "TS_PCT"
]

players = players.dropna(subset=model_columns)

print("Players eligible for modeling:", len(players))

display(players[
    ["player_name", "salary_millions",
     "points_per_game", "minutes_per_game"]
].head())

Players eligible for modeling: 270


,player_name,salary_millions,points_per_game,minutes_per_game
1,AJ Green,2.120693,7.4,22.7
3,Aaron Gordon,22.841455,14.7,28.4
5,Aaron Nesmith,11.000000,12.0,24.9
6,Aaron Wiggins,10.514017,12.0,22.9
11,Al Horford,9.500000,9.0,27.6


In [4]:
# Statistics to convert to per-36-minute values
stats_to_convert = {
    "points_per_game": "pts_per36",
    "rebounds_per_game": "reb_per36",
    "assists_per_game": "ast_per36",
    "steals_per_game": "stl_per36",
    "blocks_per_game": "blk_per36"
}

for original, new in stats_to_convert.items():
    players[new] = (
        players[original] /
        players["minutes_per_game"] * 36
    )

# Preview per-36-minute statistics
display(
    players[
        [
            "player_name",
            "minutes_per_game",
            "pts_per36",
            "reb_per36",
            "ast_per36",
            "stl_per36",
            "blk_per36"
        ]
    ].head(10)
)

,player_name,minutes_per_game,pts_per36,reb_per36,ast_per36,stl_per36,blk_per36
1,AJ Green,22.7,11.735683,3.806167,2.378855,0.792952,0.158590
3,Aaron Gordon,28.4,18.633803,6.084507,4.056338,0.633803,0.380282
5,Aaron Nesmith,24.9,17.349398,5.783133,1.734940,1.156627,0.578313
6,Aaron Wiggins,22.9,18.864629,6.131004,2.829694,1.257642,0.314410
11,Al Horford,27.6,11.739130,8.086957,2.739130,0.782609,1.173913
13,Alex Caruso,19.3,13.243523,5.409326,4.663212,2.984456,1.119171
17,Alex Sarr,27.1,17.269373,8.634686,3.188192,0.929889,1.992620
19,Alperen Sengun,31.5,21.828571,11.771429,5.600000,1.257143,0.914286
20,Amen Thompson,32.3,15.715170,9.139319,4.235294,1.560372,1.448916
21,Amir Coffey,24.3,14.370370,3.259259,1.629630,0.888889,0.148148


In [5]:
# Metrics used in the production model
features = [
    "pts_per36",
    "ast_per36",
    "reb_per36",
    "stl_per36",
    "blk_per36",
    "TS_PCT"
]

# Standardize all metrics
scaler = StandardScaler()

scaled_data = scaler.fit_transform(
    players[features]
)

# Convert standardized values to DataFrame
scaled_df = pd.DataFrame(
    scaled_data,
    columns=[f"{col}_z" for col in features],
    index=players.index
)

# Add standardized columns to player dataset
players = pd.concat([players, scaled_df], axis=1)

display(
    players[
        [
            "player_name",
            "pts_per36_z",
            "ast_per36_z",
            "reb_per36_z",
            "TS_PCT_z"
        ]
    ].head()
)

,player_name,pts_per36_z,ast_per36_z,reb_per36_z,TS_PCT_z
1,AJ Green,-1.078052,-0.755288,-0.962407,0.942858
3,Aaron Gordon,0.314133,0.061102,-0.183722,1.575609
5,Aaron Nesmith,0.054913,-1.068666,-0.286725,1.651775
6,Aaron Wiggins,0.360719,-0.535875,-0.167830,0.399861
11,Al Horford,-1.077357,-0.579951,0.500671,-0.311889


In [6]:
# Define metric weights
weights = {
    "pts_per36_z": 0.30,
    "ast_per36_z": 0.20,
    "reb_per36_z": 0.15,
    "stl_per36_z": 0.10,
    "blk_per36_z": 0.05,
    "TS_PCT_z": 0.20
}

# Check weights sum to 100%
assert np.isclose(sum(weights.values()), 1.0)

# Calculate weighted production score
players["production_score"] = sum(
    players[column] * weight
    for column, weight in weights.items()
)

# Show highest production scores
display(
    players[
        [
            "player_name",
            "production_score",
            "salary_millions",
            "points_per_game"
        ]
    ].sort_values(
        "production_score",
        ascending=False
    ).head(15)
)

,player_name,production_score,salary_millions,points_per_game
423,Nikola Jokić,2.107053,51.415938,29.6
490,Shai Gilgeous-Alexander,1.677740,35.859950,32.7
180,Giannis Antetokounmpo,1.673088,48.787676,30.4
366,Luka Dončić,1.333512,43.031940,28.2
550,Victor Wembanyama,1.189553,12.768960,24.3
28,Anthony Davis,1.065319,43.219440,24.7
358,LeBron James,1.063723,48.728845,24.4
69,Cade Cunningham,1.008016,13.940809,26.1
146,Domantas Sabonis,0.981643,40.500000,19.1
115,Daniel Gafford,0.967287,13.394160,12.3


In [7]:
# Calculate minutes-based reliability factor
players["reliability_factor"] = np.minimum(
    1.0,
    np.sqrt(players["total_minutes_est"] / 2000)
)

# Calculate adjusted production
players["adjusted_production"] = (
    players["production_score"] *
    players["reliability_factor"]
)

display(
    players[
        [
            "player_name",
            "total_minutes_est",
            "production_score",
            "reliability_factor",
            "adjusted_production"
        ]
    ].head(10)
)

,player_name,total_minutes_est,production_score,reliability_factor,adjusted_production
1,AJ Green,1657.1,-0.582918,0.910247,-0.530600
3,Aaron Gordon,1448.4,0.221330,0.850999,0.188352
5,Aaron Nesmith,1120.5,0.061147,0.748498,0.045768
6,Aaron Wiggins,1740.4,0.029253,0.932845,0.027289
11,Al Horford,1656.0,-0.496231,0.909945,-0.451543
13,Alex Caruso,1042.2,0.195662,0.721873,0.141243
17,Alex Sarr,1815.7,-0.336897,0.952812,-0.320999
19,Alperen Sengun,2394.0,0.596320,1.000000,0.596320
20,Amen Thompson,2228.7,0.326014,1.000000,0.326014
21,Amir Coffey,1749.6,-0.530584,0.935307,-0.496259


In [8]:
# Convert salary to logarithmic scale
players["log_salary"] = np.log(
    players["salary_millions"]
)

display(
    players[
        ["player_name", "salary_millions", "log_salary"]
    ].head()
)

,player_name,salary_millions,log_salary
1,AJ Green,2.120693,0.751743
3,Aaron Gordon,22.841455,3.128577
5,Aaron Nesmith,11.000000,2.397895
6,Aaron Wiggins,10.514017,2.352709
11,Al Horford,9.500000,2.251292


In [9]:
# Define explanatory and response variables
X = players[["log_salary"]]
y = players["production_score"]

# Fit linear regression
model = LinearRegression()
model.fit(X, y)

# Generate expected production
players["expected_production"] = model.predict(X)

# Model diagnostics
r_squared = model.score(X, y)

print("Regression intercept:", round(model.intercept_, 4))
print("Salary coefficient:", round(model.coef_[0], 4))
print("R-squared:", round(r_squared, 4))

Regression intercept: -0.5626
Salary coefficient: 0.251
R-squared: 0.2393


In [10]:
# Calculate salary-adjusted value score
players["value_score"] = (
    players["production_score"] -
    players["expected_production"]
)

# Rank players from highest to lowest
players["value_rank"] = (
    players["value_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Display top 20 value scores
top_value = players.sort_values(
    "value_score",
    ascending=False
).head(20)

display(
    top_value[
        [
            "value_rank",
            "player_name",
            "salary_millions",
            "production_score",
            "expected_production",
            "value_score"
        ]
    ].round(3)
)

,value_rank,player_name,salary_millions,production_score,expected_production,value_score
423,1,Nikola Jokić,51.416,2.107,0.426,1.681
490,2,Shai Gilgeous-Alexander,35.860,1.678,0.336,1.342
180,3,Giannis Antetokounmpo,48.788,1.673,0.413,1.260
538,4,Ty Jerome,2.561,0.853,-0.327,1.180
550,5,Victor Wembanyama,12.769,1.190,0.077,1.113
227,6,Jalen Duren,4.537,0.874,-0.183,1.057
383,7,Mark Williams,4.094,0.846,-0.209,1.055
366,8,Luka Dončić,43.032,1.334,0.382,0.952
69,9,Cade Cunningham,13.941,1.008,0.099,0.909
487,10,Scotty Pippen Jr.,2.088,0.519,-0.378,0.897


In [11]:
# Scatter plot: Salary vs. production
fig = px.scatter(
    players,
    x="salary_millions",
    y="production_score",
    color="value_score",
    hover_name="player_name",
    hover_data=[
        "value_rank",
        "expected_production",
        "points_per_game"
    ],
    color_continuous_scale="RdYlGn",
    title="NBA Player Production vs. Salary (2024–25)",
    labels={
        "salary_millions": "Salary ($ Millions)",
        "production_score": "Production Score",
        "value_score": "Value Score"
    },
    template="plotly_white"
)

# Create predicted production curve
salary_grid = np.linspace(
    players["salary_millions"].min(),
    players["salary_millions"].max(),
    200
)

prediction_grid = pd.DataFrame({
    "log_salary": np.log(salary_grid)
})

expected_line = model.predict(prediction_grid)

fig.add_scatter(
    x=salary_grid,
    y=expected_line,
    mode="lines",
    name="Expected production",
    line=dict(color="black", width=2)
)

fig.update_layout(height=600)

fig.show()

In [12]:
top15 = players.nlargest(15, "value_score").sort_values(
    "value_score",
    ascending=True
)

fig = px.bar(
    top15,
    x="value_score",
    y="player_name",
    orientation="h",
    color="value_score",
    color_continuous_scale="Greens",
    title="Top 15 Salary-Adjusted NBA Players (2024–25)",
    labels={
        "player_name": "Player",
        "value_score": "Value Score"
    },
    template="plotly_white"
)

fig.update_layout(
    height=650,
    showlegend=False,
    coloraxis_showscale=False
)

fig.show()

Part 3 — Test whether our model is reasonable
This is where we can make the project more impressive from a data science perspective.
Rather than simply creating a formula and presenting the results, we should examine how sensitive the rankings are to our modeling decisions.
Step 14 — Compare with a different weighting system
Let's test an alternative weighting scheme that places slightly more emphasis on efficiency and less emphasis on scoring.

In [13]:
# Alternative production weights
alternative_weights = {
    "pts_per36_z": 0.20,
    "ast_per36_z": 0.20,
    "reb_per36_z": 0.15,
    "stl_per36_z": 0.10,
    "blk_per36_z": 0.10,
    "TS_PCT_z": 0.25
}

assert np.isclose(sum(alternative_weights.values()), 1.0)

players["production_alt"] = sum(
    players[column] * weight
    for column, weight in alternative_weights.items()
)

# Fit a new salary regression for alternative production
alternative_model = LinearRegression()

alternative_model.fit(
    players[["log_salary"]],
    players["production_alt"]
)

players["value_score_alt"] = (
    players["production_alt"] -
    alternative_model.predict(players[["log_salary"]])
)

# Compare original and alternative rankings
players["rank_alt"] = (
    players["value_score_alt"]
    .rank(method="min", ascending=False)
    .astype(int)
)

players["rank_change"] = (
    players["value_rank"] - players["rank_alt"]
)

display(
    players[
        [
            "player_name",
            "value_rank",
            "rank_alt",
            "rank_change"
        ]
    ].sort_values("value_rank").head(20)
)

,player_name,value_rank,rank_alt,rank_change
423,Nikola Jokić,1,1,0
490,Shai Gilgeous-Alexander,2,5,-3
180,Giannis Antetokounmpo,3,6,-3
538,Ty Jerome,4,8,-4
550,Victor Wembanyama,5,3,2
227,Jalen Duren,6,2,4
383,Mark Williams,7,7,0
366,Luka Dončić,8,15,-7
69,Cade Cunningham,9,16,-7
487,Scotty Pippen Jr.,10,13,-3


In [14]:
# Spearman correlation between rankings
rank_corr = players["value_rank"].corr(
    players["rank_alt"],
    method="spearman"
)

print(
    "Spearman rank correlation:",
    round(rank_corr, 3)
)

Spearman rank correlation: 0.971


In [15]:
# Confirm unique player records
assert players["player_id"].is_unique

# Confirm each salary is valid
assert (players["salary_2024_25"] > 0).all()

# Confirm no missing model outputs
assert players[
    [
        "production_score",
        "expected_production",
        "value_score",
        "value_rank"
    ]
].notna().all().all()

# Confirm value score calculation
assert np.allclose(
    players["value_score"],
    players["production_score"] -
    players["expected_production"]
)

# Confirm standardized metrics are centered near zero
for feature in features:
    assert np.isclose(
        players[f"{feature}_z"].mean(),
        0,
        atol=1e-10
    )

print("All model validation checks passed!")

All model validation checks passed!


In [16]:
# Columns needed for the dashboard
export_columns = [
    "player_id",
    "player_name",
    "team",
    "age",
    "games_played",
    "total_minutes_est",
    "minutes_per_game",
    "points_per_game",
    "rebounds_per_game",
    "assists_per_game",
    "steals_per_game",
    "blocks_per_game",
    "TS_PCT",
    "salary_2024_25",
    "salary_millions",
    "production_score",
    "expected_production",
    "value_score",
    "value_rank",
    "value_score_alt",
    "rank_alt"
]

rankings = players[export_columns].copy()

rankings = rankings.sort_values(
    "value_rank"
)

output_path = (
    project_dir /
    "data/processed/nba_player_value_rankings.csv"
)

rankings.to_csv(output_path, index=False)

print("Saved rankings to:", output_path)
print("Players exported:", len(rankings))

display(rankings.head(10))

Saved rankings to: /Users/charlieellis/nba-player-value-analyzer/data/processed/nba_player_value_rankings.csv
Players exported: 270


,player_id,player_name,team,age,games_played,total_minutes_est,minutes_per_game,points_per_game,rebounds_per_game,assists_per_game,...,blocks_per_game,TS_PCT,salary_2024_25,salary_millions,production_score,expected_production,value_score,value_rank,value_score_alt,rank_alt
423,203999,Nikola Jokić,DEN,30.0,70,2569.0,36.7,29.6,12.7,10.2,...,0.6,0.662745,51415938.0,51.415938,2.107053,0.426431,1.680621,1,1.594300,1
490,1628983,Shai Gilgeous-Alexander,OKC,26.0,76,2599.2,34.2,32.7,5.0,6.4,...,1.0,0.636806,35859950.0,35.859950,1.677740,0.335980,1.341760,2,1.141811,5
180,203507,Giannis Antetokounmpo,MIL,30.0,67,2291.4,34.2,30.4,11.9,6.5,...,1.2,0.624509,48787676.0,48.787676,1.673088,0.413260,1.259828,3,1.126419,6
538,1629660,Ty Jerome,CLE,27.0,70,1393.0,19.9,12.5,2.5,3.4,...,0.0,0.642640,2560975.0,2.560975,0.853446,-0.326537,1.179983,4,1.021394,8
550,1641705,Victor Wembanyama,SAS,21.0,46,1527.2,33.2,24.3,11.0,3.7,...,3.8,0.593516,12768960.0,12.768960,1.189553,0.076769,1.112784,5,1.241676,3
227,1631105,Jalen Duren,DET,21.0,78,2035.8,26.1,11.8,10.3,2.7,...,1.1,0.703470,4536840.0,4.536840,0.874162,-0.182989,1.057152,6,1.245343,2
383,1631109,Mark Williams,CHA,23.0,44,1170.4,26.6,15.3,10.2,2.5,...,1.2,0.646784,4094280.0,4.094280,0.845795,-0.208755,1.054549,7,1.095982,7
366,1629029,Luka Dončić,LAL,26.0,50,1770.0,35.4,28.2,8.2,7.7,...,0.4,0.587254,43031940.0,43.031940,1.333512,0.381747,0.951765,8,0.767247,15
69,1630595,Cade Cunningham,DET,23.0,70,2450.0,35.0,26.1,6.1,9.1,...,0.8,0.564885,13940809.0,13.940809,1.008016,0.098810,0.909206,9,0.723248,16
487,1630590,Scotty Pippen Jr.,MEM,24.0,79,1682.7,21.3,9.9,3.3,4.4,...,0.4,0.582003,2087519.0,2.087519,0.519209,-0.377849,0.897058,10,0.838970,13


## Model Methodology

### Production Score
Six performance metrics were standardized using z-scores:
points, assists, rebounds, steals, and blocks per 36 minutes,
plus True Shooting Percentage.

The standardized metrics were combined using a weighted
average to create a production score.

### Salary Adjustment
Player salaries were transformed using natural logarithms
to account for the right-skewed salary distribution.

A linear regression model estimated production score
based on log salary.

### Player Value Score
Value Score = Actual Production - Expected Production

Positive scores indicate above-expected statistical
production relative to salary.

### Sensitivity Analysis
An alternative weighting scheme was tested to determine
whether player rankings changed when different performance
metrics received greater emphasis.

## Model Limitations

1. Weights are subjective and are not an established NBA metric.
2. Per-36-minute statistics do not fully account for player roles.
3. Box-score statistics do not capture all defensive contributions.
4. Salary depends on contract timing, tenure, and league rules.
5. Rookie contracts can make high-performing young players appear
   especially valuable.
6. Salary figures and historical team labels require final review.
7. This is a descriptive in-sample comparison, not a model
   validated to predict future player performance.

## Conclusion

This model provides a transparent framework for evaluating
NBA player statistical production relative to compensation.

The results should be interpreted as exploratory measures of
salary-adjusted production rather than definitive assessments
of player quality or contract value.